# Question-2: CycleGAN for Person Face Sketches
Unpaired photo ↔ sketch translation (Zhu et al., 2017), trained end-to-end on
[Person Face Sketches](https://www.kaggle.com/datasets/almightyj/person-face-sketches).

**Runtime → Change runtime type → GPU.** Checkpoints go to Google Drive. After a disconnect, run the cells again from the top and training picks up from the last saved epoch.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available())

## 2. Drive + code
Upload this `q2/` folder to `MyDrive/as2/q2` first. If you use a different location, change `CODE_DIR`.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
CODE_DIR = '/content/drive/MyDrive/as2/q2'
CKPT = '/content/drive/MyDrive/cyclegan_ckpt'
%cd $CODE_DIR
!pip -q install -r requirements.txt

## 3. Dataset
`kagglehub` asks for your Kaggle credentials if it needs them. The second command prints how many photos and sketches the loader found. If it picked the wrong folders, set `PHOTO_DIR`/`SKETCH_DIR` below.

In [ ]:
import kagglehub
DATA = kagglehub.dataset_download("almightyj/person-face-sketches")
print(DATA)
!find "$DATA" -maxdepth 3 -type d | head -20
!python dataset.py --data_dir "$DATA"

In [ ]:
# Leave empty to auto-split by folder name, or set explicit folders, e.g. f"{DATA}/train/photos"
PHOTO_DIR, SKETCH_DIR = "", ""
DATA_FLAGS = f'--data_dir "{DATA}"' + (f' --photo_dir "{PHOTO_DIR}" --sketch_dir "{SKETCH_DIR}"' if PHOTO_DIR and SKETCH_DIR else "")
print(DATA_FLAGS)

## 4. Hyperparameter tuning (short runs)
Each setting below trains for 3 epochs at 128 px on 1,000 images per domain, with its own checkpoint dir. We compare
the cycle-consistency loss (lower means better reconstruction) and look at the sample grids. GAN losses alone don't say much about quality.

In [ ]:
TUNE = {
    "lr2e-4_bs1": "--lr 2e-4 --batch_size 1",
    "lr1e-4_bs1": "--lr 1e-4 --batch_size 1",
    "lr2e-4_bs4": "--lr 2e-4 --batch_size 4",
}
for name, flags in TUNE.items():
    !python train.py $DATA_FLAGS --checkpoint_dir "$CKPT/tune_$name" {flags} \
        --epochs 3 --image_size 128 --n_blocks 6 --max_samples 1000 --snapshot_every 0 --amp --log_every 200

In [ ]:
import pandas as pd
rows = []
for name in TUNE:
    h = pd.read_csv(f"{CKPT}/tune_{name}/history.csv")
    rows.append({"run": name, **h.iloc[-1][["loss_G", "loss_D", "loss_cycle", "loss_idt", "seconds"]].to_dict()})
pd.DataFrame(rows).sort_values("loss_cycle")

In [ ]:
from IPython.display import Image, display
for name in TUNE:
    print(name); display(Image(f"{CKPT}/tune_{name}/samples/epoch_003.png"))

## 5. Full training
Put the winning `--lr`/`--batch_size` in `BEST`. The run checkpoints every epoch, so if Colab disconnects, re-run this cell (and the cells above it) and it resumes from `latest.pth`. If you run out of memory, try `--image_size 128 --n_blocks 6`, a larger `--batch_size` with a smaller image size, or `--max_samples`.

In [ ]:
BEST = "--lr 2e-4 --batch_size 1"
!python train.py $DATA_FLAGS --checkpoint_dir "$CKPT/main" {BEST} --epochs 40 --image_size 256 --amp --num_workers 2

## 6. Monitor training

In [ ]:
import glob
h = pd.read_csv(f"{CKPT}/main/history.csv")
h.plot(x="epoch", y=["loss_G", "loss_D", "loss_cycle", "loss_idt"], figsize=(9, 4), title="CycleGAN losses")
display(h.tail())
latest = sorted(glob.glob(f"{CKPT}/main/samples/epoch_*.png"))[-1]
print(latest, "(rows: photo, ->sketch, sketch, ->photo)"); display(Image(latest))

## 7. Test both directions
Uses the auto-detector: a photo becomes a sketch and a sketch becomes a photo.

In [ ]:
from dataset import resolve_domains
photos, sketches = resolve_domains(DATA, PHOTO_DIR or None, SKETCH_DIR or None)
for i, src in enumerate([photos[-1], sketches[-1]]):
    !python inference.py --checkpoint "$CKPT/main/latest.pth" --input "{src}" --output /content/out_{i}.png
    display(Image(str(src), width=256), Image(f"/content/out_{i}.png", width=256))

## 8. Web UI
Starts Flask in the background and prints the proxied HTTPS URL. Open it in a new tab: upload, camera and live mode all work there.

In [ ]:
!pkill -f "python app.py" || true
!nohup python app.py --checkpoint "$CKPT/main/latest.pth" --port 5000 > app.log 2>&1 &
import time; time.sleep(5); print(open("app.log").read()[-500:])
from google.colab.output import eval_js; print(eval_js("google.colab.kernel.proxyPort(5000)"))